[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BartekDaniluk/Big_Data/blob/main/Python_type_validation.ipynb)


# Walidacja typów danych w pythonie

In [1]:
string: str = 'Jakiś string'
integer: int = 1234

In [2]:
def create_car(brand: str, model: str = None, displacement: float | None = None) -> dict[str, str | float | None]:
  return {
      'brand': brand,
      'model': model,
      'displacement': displacement
  }

In [3]:
create_car('Ford', 'Mustang boss 429', 7.0)

{'brand': 'Ford', 'model': 'Mustang boss 429', 'displacement': 7.0}

In [4]:
type Car = dict[str, str | float | None]

def create_car(brand: str, model: str = None, displacement: float | None = None) -> Car:
  return {
      'brand': brand,
      'model': model,
      'displacement': displacement
  }

In [5]:
create_car('Nissan', 'Skyline R32', 3.0)

{'brand': 'Nissan', 'model': 'Skyline R32', 'displacement': 3.0}

In [6]:
from typing import NewType

four_speed_gear_ratios = NewType('four_speed_gear_ratios', tuple[float, float, float, float, float])
type Car = dict[str, str | float | four_speed_gear_ratios | None]

def create_car(brand: str, model: str, gear_ratios: four_speed_gear_ratios, displacement: float | None = None) -> Car:
  return {
      'brand': brand,
      'model': model,
      'gear_ratios': gear_ratios,
      'displacement': displacement
  }

create_car('Ford', 'Mustang boss 429', four_speed_gear_ratios((2.32, 1.69, 1.29, 1.0 , 2.32)), 7.0)

{'brand': 'Ford',
 'model': 'Mustang boss 429',
 'gear_ratios': (2.32, 1.69, 1.29, 1.0, 2.32),
 'displacement': 7.0}

In [7]:
from typing import NewType, TypedDict

four_speed_gear_ratios = NewType('four_speed_gear_ratios', tuple[float, float, float, float, float])

class Car(TypedDict):
  brand: str
  model: str
  gear_ratio: four_speed_gear_ratios
  displacement: float | None

def create_car(brand: str, model: str, gear_ratios: four_speed_gear_ratios, displacement: float | None = None) -> Car:
  return {
      'brand': brand,
      'model': model,
      'gear_ratios': gear_ratios,
      'displacement': displacement
  }
create_car('Ford', 'Mustang boss 429', four_speed_gear_ratios((2.32, 1.69, 1.29, 1.0 , 2.32)), 7.0)

{'brand': 'Ford',
 'model': 'Mustang boss 429',
 'gear_ratios': (2.32, 1.69, 1.29, 1.0, 2.32),
 'displacement': 7.0}

### Dataclasses – "Standardowe" podejście do kontenerów danych

Wprowadzone w Pythonie 3.7, `dataclasses` automatyzują tworzenie metod takich jak `__init__`, `__repr__` czy `__eq__`. W przeciwieństwie do `TypedDict`, tworzą one prawdziwą klasę (obiekt), a nie tylko podpowiedź dla słownika.

In [8]:
from dataclasses import dataclass

@dataclass
class CarData:
    brand: str
    model: str
    gear_ratios: four_speed_gear_ratios
    displacement: float | None = None

car_obj = CarData('Ford', 'Mustang boss 429', four_speed_gear_ratios((2.32, 1.69, 1.29, 1.0, 2.32)), 7.0)
print(car_obj)

CarData(brand='Ford', model='Mustang boss 429', gear_ratios=(2.32, 1.69, 1.29, 1.0, 2.32), displacement=7.0)


### Dataclasses — co jeszcze potrafią?

`@dataclass` to prawdziwa kopalnia złota. Domyślnie generuje `__init__`, `__repr__` i `__eq__`, ale to dopiero początek. Dekorator przyjmuje masę parametrów które zmieniają zachowanie klasy:

| Parametr | Domyślnie | Co robi |
| :--- | :---: | :--- |
| `init` | `True` | Generuje `__init__` |
| `repr` | `True` | Generuje `__repr__` |
| `eq` | `True` | Porównanie po wartościach pól, nie po identyczności obiektu |
| `order` | `False` | Generuje `__lt__`, `__le__`, `__gt__`, `__ge__` — możesz sortować listę dataclassów |
| `frozen` | `False` | Robi instancję niemutowalną — próba `obj.x = 1` rzuca `FrozenInstanceError` |
| `slots` (Python 3.10+) | `False` | Bardzo wydajna opcja — alokuje stały blok pamięci zamiast `__dict__` |
| `unsafe_hash` | `False` | Wymusza generację `__hash__` (normalnie jest tylko gdy `frozen=True`) |

> **Wskazówka Big Data:** `slots=True` to game-changer przy dużej liczbie obiektów. Standardowe klasy w Pythonie trzymają atrybuty w słowniku `__dict__`, co pożera ~280 bajtów na instancję. `slots=True` alokuje statyczną tablicę pamięci dla z góry znanej listy pól. Przy milionie instancji to różnica między ~300 MB a ~50 MB. Crucial przy symulacjach numerycznych, parsowaniu logów, wysokoczęstotliwościowym tradingu, czy stanie grafu w LangGraphie.

Pokażmy wszystko naraz — frozen, order, slots, `__post_init__` i `InitVar`:


In [ ]:
from dataclasses import dataclass, field, InitVar

@dataclass(frozen=True, order=True, slots=True)
class ComputeTask:
    # sort_index nie pojawia się w __init__, ale steruje porządkiem (order=True)
    sort_index: float = field(init=False, repr=False)

    task_name: str
    base_cost: float = 10.5

    # InitVar — pojawia się w __init__, ale NIE zostaje jako atrybut instancji
    priority_multiplier: InitVar[float] = 1.0

    # Listy/słowniki MUSZĄ używać default_factory (mutable defaults!)
    metadata: dict[str, str] = field(default_factory=dict)

    def __post_init__(self, priority_multiplier: float):
        # frozen blokuje normalne przypisanie, ale wewnątrz __post_init__
        # używamy obejścia object.__setattr__
        object.__setattr__(self, 'sort_index', self.base_cost * priority_multiplier)

t1 = ComputeTask("Train model", base_cost=100, priority_multiplier=2.0)
t2 = ComputeTask("Save checkpoint", base_cost=50, priority_multiplier=3.0)
print(t1)
print("Posortowane po sort_index:", sorted([t1, t2]))


Sprawdźmy że `frozen` rzeczywiście blokuje modyfikacje:


In [ ]:
try:
    t1.task_name = "Coś nowego"
except Exception as e:
    print(f"{type(e).__name__}: {e}")


### `NamedTuple` — niemutowalny brat dataclassy

Jeśli potrzebujesz **niemutowalnego, indeksowanego** kontenera danych z dostępem przez nazwę pola, `NamedTuple` to twój człowiek. To po prostu krotka, ale z nazwami i annotacjami:


In [ ]:
from typing import NamedTuple

class Point(NamedTuple):
    x: float
    y: float
    label: str = "origin"  # default values działają jak w dataclass

p = Point(1.0, 2.0, "start")
print(p)
print("Po nazwie:", p.x, "| Po indeksie:", p[0])
print("Slicing nadal działa:", p[:2])
# p.x = 5  # AttributeError — niemutowalna


### Porównanie kontenerów danych — który wybrać?

Mamy już cały arsenał. Czas na decyzję, którą strukturę wybrać do czego:

| Struktura | Mutowalna? | Walidacja runtime? | Dostęp po nazwie? | Indeksowalność | Typowy use case |
| :--- | :---: | :---: | :---: | :---: | :--- |
| `dict` | ✅ | ❌ | ✅ (klucze) | ❌ | Dynamiczne payload-y, JSON-y |
| `TypedDict` | ✅ | ❌ (tylko statyczna) | ✅ | ❌ | Słowniki ze schematem dla type-checkera (IDE/mypy) |
| `NamedTuple` | ❌ | ❌ | ✅ | ✅ | Niemutowalne małe paczki danych (np. współrzędne) |
| `dataclass` | ✅ (chyba że `frozen`) | ❌ | ✅ | ❌ | Wewnętrzne kontenery domeny biznesowej |
| `attrs` | ✅ | Opcjonalna | ✅ | ❌ | Jak dataclass, ale z bogatszą walidacją (3rd party) |
| **Pydantic** | ✅ | ✅ (rygorystyczna) | ✅ | ❌ | **Granice systemu** — API, LLM I/O, walidacja JSON-ów |

Wszystkie te narzędzia bazują na `__annotations__` — magicznym słowniku gdzie Python trzyma type hinty. Spójrz, oba kontenery mają go automatycznie:


In [ ]:
print("ComputeTask:", ComputeTask.__annotations__)
print("Point:       ", Point.__annotations__)


### Nominalne vs Strukturalne Typowanie (Duck Typing)

Python tradycyjnie opiera się na "Duck Typing" (jeśli kwacze jak kaczka, to jest kaczką). 
- **Typowanie Nominalne** (np. `class Car(Vehicle)`): Obiekt musi jawnie dziedziczyć po danej klasie.
- **Typowanie Strukturalne** (`typing.Protocol`): Obiekt musi po prostu posiadać wymagane metody i atrybuty. To doskonałe narzędzie do tworzenia elastycznych interfejsów bez wymuszania hierarchii klas.

In [9]:
from typing import Protocol

class Movable(Protocol):
    def move(self) -> None: ...

class Robot:
    def move(self) -> None:
        print("Robot jedzie...")

class Bird:
    def move(self) -> None:
        print("Ptak leci...")

def start_movement(obj: Movable):
    obj.move()

start_movement(Robot())
start_movement(Bird())

Robot jedzie...
Ptak leci...


# Pydantic

In [10]:
import pydantic

pydantic.__version__

'2.7.1'

## Pydantic V2 — co siedzi pod spodem?

Zanim zaczniemy używać Pydantica do walidacji, warto wiedzieć **dlaczego** jest tak szybki. Pydantic V2 (wydany w 2023) to gigantyczny przepis architekturalny względem V1:

* **`pydantic-core`** — cały silnik walidacji i serializacji jest napisany w **Ruście**, nie w Pythonie. Python tylko definiuje modele.
* **Core Schema** — kiedy definiujesz `BaseModel`, metaklasa analizuje annotacje i buduje "blueprint" walidacji w postaci słownika. Ten blueprint trafia do Rusta przez FFI (Foreign Function Interface).
* **Wynik:** 5x — 20x szybsza walidacja względem V1. Pydantic V2 jest porównywalny z dedykowanymi parserami JSON.

Schematycznie:

```
[Twój kod Python]
   ↓ definicja BaseModel
[Metaklasa Pydantica] → [GenerateSchema]
   ↓ core_schema (dict)
[__pydantic_core_schema__]  ← ten atrybut jest na każdej klasie
   ↓ FFI
[Rust: pydantic-core]
   ↓ walidacja, koercja, serializacja
[Twój zwalidowany obiekt]
```

> **Wskazówka Big Data:** Pydantic płaci tzw. *validation tax* — narzut wynikający z tego, że przy każdej instancjacji obiekt musi przejść przez całą rurkę walidacji w Ruście. To wciąż szybkie, ale **nie** dorównuje gołym dataclassom przy operacjach typu "milion obiektów na sekundę". W systemach Big Data Pydantic stosujemy na **brzegach** systemu (ingress/egress, walidacja JSON-ów z Kafki, walidacja outputów LLM), a wewnątrz pipeline'u używamy lekkich dataclassów albo NumPy/Pandas. Wrócimy do tego pod koniec notebooka.


W przeciwieństwie do normalengo pythonowskiego typehintingu, obiekty dziedziczące po pydanticowym BaseModel, są sprawdzane podczas runtime, w kontkeście zgodności danych. Wszystko musie się zgadzać.

In [11]:
from datetime import datetime
from pydantic import BaseModel, ValidationError

class User(BaseModel):
    uid: int
    username: str
    date_of_birth: datetime | None = None
    is_active: bool = True

    first_name: str | None = None
    surname: str | None = None

In [12]:
u1 = User(
    uid = 112,
    username = "Username_1",
    first_name = "Name1",
    surname = "Surname1"
)

Jeśli coś się nie zgadza, od razu rzucany jest ValidationError, który można ładnie przechwycić. Z ciekawostek pydantic ma domyslnie włączone typeconversion. Czyli jeśli w int wrzucimu '123', nit będzie problemu, zostanie to zamieniane na int. Poód do tego jest taki że pozyskiwanie numerycznych wartości, w postaci stringów, jest dośc popularne. Czy to z jsona, parametrów URL itd...

In [13]:
try:
    u12 = User(
        uid = '1123',
        username = 123123,
        first_name = "Name1",
        surname = "Surname1"
    )
except ValidationError as e:
    print(e)

1 validation error for User
username
  Input should be a valid string [type=string_type, input_value=123123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.7/v/string_type


In [14]:
print(u1)

uid=112 username='Username_1' date_of_birth=None is_active=True first_name='Name1' surname='Surname1'


Wspiera również natywną serializację.

In [15]:
u1.model_dump_json(indent=2)

'{\n  "uid": 112,\n  "username": "Username_1",\n  "date_of_birth": null,\n  "is_active": true,\n  "first_name": "Name1",\n  "surname": "Surname1"\n}'

### Performance trick: `model_validate_json()` zamiast `model_validate(json.loads(...))`

Bardzo częsty antypattern — ludzie ładują JSON do dicta przez `json.loads()`, a potem przekazują dict do Pydantica:


In [ ]:
import json

raw_payload = '{"uid": 999, "username": "Bartek", "is_active": true}'

# ANTYPATTERN — Python parsuje JSON, alokuje dict, dopiero potem Pydantic to bierze
u_slow = User.model_validate(json.loads(raw_payload))

# ZALECANE — Rust parsuje JSON bezpośrednio, BEZ pośredniego dicta
u_fast = User.model_validate_json(raw_payload)

print("Identyczne:", u_slow == u_fast)


> **Wskazówka Big Data:** Przy strumieniach JSON-ów z Kafki czy Spark Structured Streamingu, ten jeden trick może dać kilkadziesiąt procent mniej alokacji pamięci. Liczy się każda mikro-sekunda kiedy walidujesz miliony eventów dziennie. Generalna zasada: jeśli masz string JSON-a — używaj `model_validate_json()`. Jeśli masz już dict (np. z innej biblioteki) — `model_validate()`. Nigdy nie rób `model_validate(json.loads(x))` ręcznie.


### Dlaczego `default_factory`? Problem Mutable Defaults

W Pythonie domyślne argumenty funkcji (i pól klas) są ewaluowane **tylko raz** – w momencie definicji. Jeśli użyjemy obiektu mutowalnego (np. `list`, `dict`), ten sam obiekt będzie dzielony między wszystkie instancje klasy.

**Przykład błędu:**
```python
def add_to_list(val, my_list=[]): # To samo [] dla każdego wywołania!
    my_list.append(val)
    return my_list
```

`default_factory` rozwiązuje ten problem, przyjmując funkcję, która zostanie wywołana za każdym razem, gdy potrzebna jest nowa domyślna wartość.

### Czym jest `functools.partial`?

`partial` to funkcja wyższego rzędu, która pozwala na "zamrożenie" części argumentów innej funkcji, tworząc nowy obiekt wywoływalny o uproszczonej sygnaturze. Jest to niezwykle przydatne właśnie w `default_factory`, gdy chcemy przekazać parametry do funkcji tworzącej (np. strefę czasową do `datetime.now`). Albo generowanie unikatowego id za pomocą uuid

In [16]:
from pydantic import Field
import pytz
from functools import partial
from typing import Literal
from uuid import UUID, uuid4

class SupportTicket(BaseModel):
    uid: UUID = Field(default_factory=uuid4)
    sender_id: int
    content: str
    priority: Literal['low', 'medium', 'high'] = 'low'
    is_resolved: bool = False

    tages: list[str] = Field(default_factory=list)
    created_at: datetime = Field(default_factory=partial(datetime.now, pytz.timezone('Europe/Berlin')))

    status: Literal['unresolved', 'being resolved', 'resolved'] = 'unresolved'

In [17]:
 st1 = SupportTicket(
    sender_id = u1.uid,
    content = "Test content",
    tags = ['tag1', 'tag2'],
    status = 'being resolved'
 )

In [18]:
st1.model_dump_json(indent=2)

'{\n  "uid": "9996ef08-c1ec-4ae4-94fa-0197fc4f3b1f",\n  "sender_id": 112,\n  "content": "Test content",\n  "priority": "low",\n  "is_resolved": false,\n  "tages": [],\n  "created_at": "2026-04-14T01:46:37.820025+02:00",\n  "status": "being resolved"\n}'

Jeśłi chodzi o Field, to ma wiele więcej możłiwości. Możdna dodawać wartości min max, długośc, wzory (regexy) itd... Zeby to zaimplementować, wykorzystujemy type Annotated z pythonowskiego typing

In [19]:
from typing import Annotated
from pydantic import Field

class User(BaseModel):
    uid: UUID = Field(default_factory=uuid4)
    email: Annotated[str, Field(pattern=r"^[^@]+@[^@]+\\.[^@]+$")]
    username: Annotated[str, Field(min_length=6, max_length=30)]
    date_of_birth: datetime | None = None
    is_active: bool = True

    first_name: str | None = None
    surname: str | None = None

In [20]:
try:
    u = User(
        email = 'mailgmail.com',
        password = 'Haslo!23',
        users_url = 'https://www.myfitnesspal.com/food/diary/mail123',
        username = 'MisioPysio69',
    )
except ValidationError as e:
    print(e)

1 validation error for User
email
  String should match pattern '^[^@]+@[^@]+\\.[^@]+$' [type=string_pattern_mismatch, input_value='mailgmail.com', input_type=str]
    For further information visit https://errors.pydantic.dev/2.7/v/string_pattern_mismatch


Czyli można sprawdzać maile własnoręcze, ale po co się męczyć? Pydantic może to zrobić za nas. Np, z Emailami, Hasłem, URL.

In [21]:
from pydantic import HttpUrl, SecretStr, EmailStr

class User(BaseModel):
    uid: UUID = Field(default_factory=uuid4)
    email: EmailStr
    password: SecretStr
    users_url: HttpUrl | None
    username: Annotated[str, Field(min_length=6, max_length=30)]
    date_of_birth: datetime | None = None
    is_active: bool = True

    first_name: str | None = None
    surname: str | None = None

In [22]:
try:
    u = User(
        email = 'mail123@gmail.com',
        password = 'Haslo!23',
        users_url = 'https://www.myfitnesspal.com/food/diary/mail123',
        username = 'MisioPysio69',
    )
except ValidationError as e:
    print(e)

In [23]:
print(u)

uid=UUID('0107cfad-26cf-46c0-8d29-49df498d3324') email='mail123@gmail.com' password=SecretStr('**********') users_url=Url('https://www.myfitnesspal.com/food/diary/mail123') username='MisioPysio69' date_of_birth=None is_active=True first_name=None surname=None


In [24]:
try:
    u = User(
        email = 'krzysiukonweka',
        password = 'Haslo!23',
        users_url = 'https://www.myfitnesspal.com/food/diary/krzysiu123',
        username = 'MisioPysio69',
    )
except ValidationError as e:
    print(e)

1 validation error for User
email
  value is not a valid email address: An email address must have an @-sign. [type=value_error, input_value='krzysiukonweka', input_type=str]


Jak bardzo chcemy, to możemy sprawdfzić secret.

In [25]:
print(u.password.get_secret_value())

Haslo!23


Custom validators

czasami wbudowana walidacja pól pydantica jest niewystarczająca, i potrzeba nam czegoś więcej. Np. username ma używać tylko alfanumerycznych znaków, z underscoreami, i tylko angielskimi literami. Tutaj mamy after validation, czyli nasza validacja działa po domyślnej walidacji pydanticowej.

In [26]:
from pydantic import field_validator, model_validator, ValidationInfo
import re

class User(BaseModel):
    uid: UUID = Field(default_factory=uuid4)
    email: EmailStr | None = None
    password: SecretStr | None = None
    users_url: HttpUrl | None
    username: Annotated[str, Field(min_length=6, max_length=30)]
    date_of_birth: datetime | None = None
    is_active: bool = True

    first_name: str | None = None
    surname: str | None = None

    @field_validator("username")
    @classmethod
    def validate_username(cls, v: str) -> str:
        if not re.match(r'^[a-zA-Z0-9_]+$', v):
            raise ValueError('Username must be alphanumeric(Eng. Letters only), underscores allowed')
        return v

In [27]:
try:
    print(User(username="valid_user123")) 
    print(User(username="czość"))        
except ValidationError as e:
    print(e)

1 validation error for User
users_url
  Field required [type=missing, input_value={'username': 'valid_user123'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.7/v/missing


Ale czasamy bysmy mogli chceć zrobić coś jeszcze przed walidacją pydanticową, np dodać https do url, jeśli go tam nie ma, a jeśłi jest samo http, to wyrzucamy błąd. Bo jesteśmi bezpeiczne chłopaki/dziweczyny

In [28]:
from pydantic import field_validator, model_validator, ValidationInfo
import re

class User(BaseModel):
    uid: UUID = Field(default_factory=uuid4)
    email: EmailStr | None = None
    password: SecretStr | None = None
    users_url: HttpUrl | None
    username: Annotated[str, Field(min_length=6, max_length=30)]
    date_of_birth: datetime | None = None
    is_active: bool = True

    first_name: str | None = None
    surname: str | None = None

    @field_validator("username")
    @classmethod
    def validate_username(cls, v: str) -> str:
        if not re.match(r'^[a-zA-Z0-9_]+$', v):
            raise ValueError('Username must be alphanumeric(Eng. Letters only), underscores allowed')
        return v

    @field_validator("users_url", mode='before')
    @classmethod
    def validate_and_add_https(cls, v: str) -> str:
        if v and not v.startswith('https://') or v.startswith('http://'):
            return f'https://{v}'
        elif v.startswith('http://'):
            raise ValueError('url must be https')
        else:
            return v

In [29]:
try:
    print(User(username="valid_user123", users_url='https://jakis_tam_url')) 
    print(User(username="valid_user123", users_url='http://jakis_tam_url'))        
except ValidationError as e:
    print(e)

uid=UUID('c92ea7ae-38e4-4c45-b563-b01c05a8c8e7') email=None password=None users_url=Url('https://jakis_tam_url/') username='valid_user123' date_of_birth=None is_active=True first_name=None surname=None
uid=UUID('68011573-9928-4888-a5ed-ff304b133887') email=None password=None users_url=Url('https://http//jakis_tam_url') username='valid_user123' date_of_birth=None is_active=True first_name=None surname=None


### Tryby walidatorów — pełna lista

Pełna lista trybów `@field_validator`:

| Tryb | Kiedy się wykonuje | Typowy użycie |
| :--- | :--- | :--- |
| `after` (domyślny) | **Po** walidacji Pydantica. Dane są już prawidłowego typu. | Reguły biznesowe (np. username musi być alfanumeryczny) |
| `before` | **Przed** walidacją Pydantica. Dostajesz surowy input. | Normalizacja (np. dodanie `https://`, strip whitespace, lowercase) |
| `wrap` | Owija walidację — możesz odpalić kod przed I po, oraz złapać błędy walidacji. | Zaawansowane — fallback values, retry logic |
| `plain` | **Zamiast** walidacji Pydantica. Pełna manualna kontrola — Pydantic nie tknie pola. | Kiedy w 100% wiesz co robisz (rzadko stosowane) |

### `ValidationInfo` — uwaga na cross-field access w `field_validator`

W Pydantic V2 walidator pola może dostać drugi argument `info: ValidationInfo`, przez który masz dostęp do `info.data` (już zwalidowane pola). **Ale uwaga** — pola walidują się w kolejności definicji w klasie. Pole które jest niżej w deklaracji może jeszcze nie być w `info.data` w momencie kiedy walidujesz pole wyżej!

Dlatego do walidacji wymagającej kilku pól używaj `@model_validator(mode='after')` — który widzi już cały zwalidowany model. To bezpieczny i zalecany wzorzec (znamy go z poprzedniego rozdziału — pamiętacie `confirm_password`?).


### Walidacja na poziomie modelu (Cross-field validation)

Czasami walidacja jednego pola zależy od wartości innego (klasyczny przykład: powtórzenie hasła). Do tego celu służy `@model_validator` z parametrem `mode='after'`.

In [30]:
class UserRegistration(BaseModel):
    username: str
    password: str = Field(..., min_length=8)
    confirm_password: str

    @model_validator(mode='after')
    def check_passwords_match(self) -> 'UserRegistration':
        if self.password != self.confirm_password:
            raise ValueError("Hasła nie są identyczne!")
        return self

try:
    reg = UserRegistration(username="student", password="tajne123", confirm_password="inne123")
except ValidationError as e:
    print(e)

1 validation error for UserRegistration
  Value error, Hasła nie są identyczne! [type=value_error, input_value={'username': 'student', '...rm_password': 'inne123'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.7/v/value_error


### Pola Obliczane (Computed Fields)

Pydantic V2 pozwala na definiowanie pól, których wartości są obliczane dynamicznie, ale zachowują się jak zwykłe pola podczas serializacji do JSONa. Wykorzystujemy do tego dekorator `@computed_field`.

In [31]:
from pydantic import computed_field

class PerformanceCar(BaseModel):
    brand: str
    model: str
    torque_lb_ft: float = Field(..., gt=0)
    max_rpm: float = Field(..., gt=0)

    @computed_field
    @property
    def horsepower(self) -> float:
        # Formuła: HP = (Torque * RPM) / 5252
        return round((self.torque_lb_ft * self.max_rpm) / 5252, 2)

car = PerformanceCar(brand="Ford", model="Mustang GT", torque_lb_ft=420, max_rpm=7000)
print(f"Moc obliczona: {car.horsepower} HP")
print(car.model_dump_json(indent=2))

Moc obliczona: 559.79 HP
{
  "brand": "Ford",
  "model": "Mustang GT",
  "torque_lb_ft": 420.0,
  "max_rpm": 7000.0,
  "horsepower": 559.79
}


### `ConfigDict` — konfigurujemy zachowanie modelu

W Pydantic V1 konfigurację robiło się przez wewnętrzną klasę `Config`. W V2 mamy `model_config: ConfigDict`. To absolutna podstawa, której często brakuje w tutorialach — a daje *masę* kontroli:


In [ ]:
from pydantic import BaseModel, ConfigDict, ValidationError

class StrictUser(BaseModel):
    model_config = ConfigDict(
        strict=True,                # wyłącza koercję typów — '123' NIE stanie się int 123
        extra='forbid',             # rzuca błąd jeśli payload zawiera pola których nie ma w modelu
        frozen=True,                # robi obiekt niemutowalnym po stworzeniu
        str_strip_whitespace=True,  # automatycznie strip()uje wszystkie stringi
    )

    uid: int
    username: str

# 1. Test stricta — string zamiast inta
try:
    StrictUser(uid='123', username='Bartek')
except ValidationError as e:
    print("STRICT  ->", e.errors()[0]['msg'])

# 2. Test extra='forbid' — nieoczekiwane pole
try:
    StrictUser(uid=1, username='Bartek', hacker_field='wstrzyknięte!')
except ValidationError as e:
    print("FORBID  ->", e.errors()[0]['msg'])

# 3. str_strip_whitespace + frozen
u = StrictUser(uid=1, username='  Bartek  ')
print("Po stripie:", repr(u.username))
try:
    u.username = 'Hacked'
except ValidationError as e:
    print("FROZEN  ->", e.errors()[0]['msg'])


> **Wskazówka Big Data:** `extra='forbid'` to **must-have** dla endpointów API. Bez tego atakujący może wcisnąć dodatkowe pola do payloadu (mass assignment / object injection — klasyk OWASP). Domyślnie Pydantic ignoruje nadmiarowe pola, co w produkcji bywa niebezpieczne.

### Pseudonimy pól (Field aliases) — gdy JSON ma kebab-case albo inne dziwactwa

Czasami zewnętrzne API zwraca brzydkie klucze typu `first-name` albo `HOST_ADDRESS`, a my chcemy pythonowskie `first_name`. Tu wchodzą aliasy:


In [ ]:
class APIPayload(BaseModel):
    model_config = ConfigDict(
        validate_by_name=True,   # akceptuje nazwę Pythonową przy wczytywaniu
        validate_by_alias=True,  # akceptuje alias przy wczytywaniu (domyślnie True)
    )

    host_address: str = Field(alias='HOST_ADDRESS')
    first_name: str = Field(alias='first-name')

# Wczytanie z brzydkiego JSON-a z zewnątrz
external = {'HOST_ADDRESS': '192.168.0.1', 'first-name': 'Bartek'}
p = APIPayload(**external)
print("Wczytany model:", p)

# Serializacja z aliasem (np. żeby odesłać w tym samym formacie do innego API)
print("Dump z aliasami:", p.model_dump(by_alias=True))

# Albo czysto pythonowsko, bez aliasów
print("Dump bez aliasów:", p.model_dump())


### `TypeAdapter` — walidacja bez `BaseModel`

Czasami chcemy zwalidować goły typ — listę intów, słownik, `TypedDict`-a — bez tworzenia całej klasy. Do tego służy `TypeAdapter`:


In [ ]:
from pydantic import TypeAdapter
from typing import TypedDict

# Walidacja listy z koercją typów
list_of_ints = TypeAdapter(list[int])
print(list_of_ints.validate_python([1, '2', 3.0]))  # → [1, 2, 3]

# Walidacja TypedDict-a — bez owijania w BaseModel
class CarPayload(TypedDict):
    brand: str
    year: int

car_validator = TypeAdapter(CarPayload)
print(car_validator.validate_python({'brand': 'Ford', 'year': '1969'}))

# Można nawet generować JSON Schema z gołych typów (przyda się dla LLM-ów!)
print("\nJSON Schema dla list[int]:")
print(list_of_ints.json_schema())


> **Wskazówka Big Data:** `TypeAdapter` **musi** być instancjowany **raz** — globalnie albo na poziomie modułu. Tworzenie go w pętli to klasyczny antypattern (każda instancja kompiluje cały schemat na nowo). Trzymaj go obok importów, jak singleton.

### Forward references i `model_rebuild()`

Jak zrobić model który odnosi się sam do siebie? Klasyczny przypadek — drzewo komentarzy, gdzie każdy komentarz ma listę odpowiedzi tego samego typu. Bez triku Python rzuca `NameError: name 'Comment' is not defined`:


In [ ]:
from __future__ import annotations  # opóźnia ewaluację annotacji do runtime

class Comment(BaseModel):
    author: str
    text: str
    replies: list['Comment'] = Field(default_factory=list)

# Bez tego Pydantic nie wie czym jest 'Comment' wewnątrz Comment
Comment.model_rebuild()

c = Comment(
    author='Bartek',
    text='Świetny notebook!',
    replies=[
        Comment(author='Kamil', text='Zgadzam się'),
        Comment(author='Miłosz', text='Też tak myślę', replies=[
            Comment(author='Bartek', text='Dzięki!')
        ])
    ]
)
print(c.model_dump_json(indent=2))


### Discriminated Unions — szybkie routowanie polimorficzne

Wyobraź sobie endpoint który przyjmuje *różne* typy zwierząt (albo eventów Kafki, albo różne tooly LLM-a). Bez triku Pydantic próbowałby zwalidować payload kolejno przeciwko każdemu typowi w `Union` — wolno i frustrujące. Discriminated Union pozwala mu wybrać właściwy schemat **natychmiastowo**, na podstawie pola-wskaźnika:


In [ ]:
from typing import Literal, Union, Annotated

class Cat(BaseModel):
    kind: Literal['cat']
    purrs_per_minute: int

class Dog(BaseModel):
    kind: Literal['dog']
    barks_loudly: bool

# Annotated + Field(discriminator='kind') — Pydantic używa pola `kind`
# jako wskaźnika do właściwego schematu
Animal = Annotated[Union[Cat, Dog], Field(discriminator='kind')]

animal_adapter = TypeAdapter(Animal)

print(animal_adapter.validate_python({'kind': 'cat', 'purrs_per_minute': 30}))
print(animal_adapter.validate_python({'kind': 'dog', 'barks_loudly': True}))

# Niezgodny dyskryminator — natychmiastowy, czytelny błąd
try:
    animal_adapter.validate_python({'kind': 'fish', 'fins': 4})
except ValidationError as e:
    print("\nBłąd:", e.errors()[0]['msg'])


> **Wskazówka Big Data:** Discriminated Union zamienia walidację z O(n) na O(1) — przy schemacie eventów Kafki gdzie typów może być kilkadziesiąt, robi to ogromną różnicę. Stosuj **wszędzie** gdzie masz polimorfizm — typy eventów, typy wiadomości, typy toolów LLM-a, etc.


# Architektura: Cykl życia danych i LangGraph

Na podstawie analizy "Python Typing vs. Pydantic Comparison", kluczowym aspektem projektowania systemów Big Data i AI jest wybór odpowiedniego narzędzia do odpowiedniego zadania. Nie zawsze Pydantic jest najlepszym wyborem.

### Data Lifecycle Heuristic Architecture (Heurystyka Cyklu Życia Danych)

| Faza Cyklu | Charakterystyka | Rekomendacja | Uzasadnienie |
| :--- | :--- | :--- | :--- |
| **Ingress (Wejście)** | Dane z zewnątrz (API, LLM, User), niepewne, surowe. | **Pydantic** | Rygorystyczna walidacja na brzegach systemu. |
| **Compute (Obliczenia)** | Wewnętrzna logika, wysoka częstotliwość, zaufane dane. | **TypedDict / Dataclasses** | Minimalny overhead, maksymalna szybkość procesora. |
| **Persist (Zapis)** | Serializacja do bazy, zachowanie stanu. | **Dataclasses** | Czystość logiczna i brak narzutu walidacji przy odczycie z zaufanego źródła. |
| **Serve (Wyjście)** | Odpowiedzi HTTP, kontrakty API. | **Pydantic** | Gwarancja zgodności z dokumentacją (OpenAPI). |

### Przypadek LangGraph: Balancing Validation and Velocity

W frameworkach agentowych takich jak **LangGraph**, stan (State) przechodzi przez dziesiątki lub setki węzłów. 
- Użycie `Pydantic` jako stanu grafu powoduje walidację całego obiektu przy każdym przejściu między węzłami (overload).
- Dlatego jako **Internal State** zaleca się `TypedDict` – daje nam to wsparcie IDE i autouzupełnianie, ale z zerowym kosztem w runtime.
- Dopiero dane wyjściowe z LLM (Structured Output) powinny przechodzić przez `Pydantic` przed umieszczeniem ich w zaufanym stanie grafu.

In [32]:
from typing import Annotated, TypedDict
import operator

# 1. WEWNĘTRZNY STAN GRAFU (Wydajność: TypedDict)
class AgentState(TypedDict):
    # Annotated tutaj służy LangGraphowi do określenia jak łączyć (reducer) wiadomości
    messages: Annotated[list[str], operator.add]
    confidence_score: float
    iteration: int

# 2. GRANICA SYSTEMU / WYJŚCIE LLM (Bezpieczeństwo: Pydantic)
class ExtractionResult(BaseModel):
    key_insights: list[str] = Field(min_length=1, description="Główne wnioski z analizy")
    confidence_score: float = Field(ge=0.0, le=1.0)

    @model_validator(mode='after')
    def verify_confidence_pairing(self) -> 'ExtractionResult':
        if len(self.key_insights) < 3 and self.confidence_score > 0.9:
            raise ValueError("Wysoka pewność wymaga przynajmniej 3 wniosków wspierających.")
        return self

print("Zaprojektowano hybrydowy system walidacji.")

Zaprojektowano hybrydowy system walidacji.


## Pydantic w erze LLM-ów — Structured Outputs

Tradycyjnie żeby wyciągnąć structured data z odpowiedzi LLM-a, trzeba było pisać kruche regexy ("znajdź pierwszą liczbę po dwukropku, załóż że to ocena"). Modern stack to porzucił — **Pydantic stał się standardem** dla LLM I/O:

* **OpenAI SDK** — natywne wsparcie dla `response_format=YourPydanticModel`
* **Anthropic SDK** — Pydantic w Tool Use
* **LangChain / LangGraph** — Pydantic dla Structured Outputs
* **Pydantic AI** — dedykowany framework od twórców Pydantica do budowy agentów (dependency injection, retry logic, observability via Logfire)

Schemat działania:

```
[Prompt do LLM-a] + [JSON Schema z Pydantica]
    ↓
[LLM generuje JSON pasujący do schematu]
    ↓
[Pydantic waliduje — czy LLM nie zhalucynował]
    ↓ (jeśli ValidationError)
[Auto-retry: błąd jako prompt → LLM próbuje znowu]
    ↓
[Bezpieczny zwalidowany obiekt w Twojej apce]
```

Definiujemy schema dla LLM-a — to ta sama klasa Pydantic-owa do której jesteśmy już przyzwyczajeni:


In [ ]:
from pydantic import BaseModel, Field, ValidationError
from typing import Literal

class MovieReview(BaseModel):
    """Strukturyzowana recenzja filmu wygenerowana przez LLM-a."""
    title: str = Field(description="Tytuł filmu")
    sentiment: Literal['positive', 'negative', 'mixed']
    rating: float = Field(ge=0.0, le=10.0, description="Ocena 0-10")
    key_points: list[str] = Field(min_length=2, description="Główne punkty recenzji")

# To wkleja się dosłownie do `response_format` w OpenAI SDK
# albo do `tools` w Anthropic SDK
import json
print(json.dumps(MovieReview.model_json_schema(), indent=2, ensure_ascii=False))


LLM dostaje informację: "musisz odpowiedzieć JSON-em pasującym **dokładnie** do tego schematu". Ale halucynuje. Często. Symulujemy sytuację, gdzie LLM zwrócił coś niewłaściwego — np. ocenę poza zakresem albo za mało key_points:


In [ ]:
# Symulowana odpowiedź z LLM-a — jakby zhalucynował
hallucinated_response = {
    "title": "The Matrix",
    "sentiment": "positive",
    "rating": 11.5,                    # poza zakresem (max 10)
    "key_points": ["Innovative effects"]  # tylko 1 punkt (min 2)
}

try:
    review = MovieReview.model_validate(hallucinated_response)
except ValidationError as e:
    # W prawdziwym pipeline'ie te błędy WRACAJĄ do LLM-a jako kolejny prompt:
    # "Twoja poprzednia odpowiedź była niepoprawna: [errors].
    #  Spróbuj jeszcze raz, tym razem trzymając się schematu."
    print("LLM zhalucynował — błędy walidacji:")
    for err in e.errors():
        print(f"  • {'.'.join(str(x) for x in err['loc'])}: {err['msg']}")


> **Wskazówka Big Data:** Ten *Validation Retry Loop* to fundament niezawodności każdego agenta AI. Bez niego LLM losowo halucynuje schemat, agent się wywala, użytkownik wkurza. **Z** Pydantic-iem agent ma szansę "naprawić się" automatycznie — ale każda próba kosztuje tokeny, więc w produkcji limituje się to do np. 3 retries. Pydantic AI ma to wbudowane out-of-the-box. LangChain też (`with_structured_output()`).

### Podsumowanie — kiedy czego używać?

Po przejściu przez cały notebook, oto reguła kciuka której trzymamy się w produkcji:

| Sytuacja | Narzędzie |
| :--- | :--- |
| Dane z API zewnętrznego, frontu, LLM-a | **Pydantic** (`BaseModel`) |
| Wewnętrzny stan grafu / pipeline-u | `TypedDict` lub `dataclass(slots=True)` |
| Niemutowalne tuple-podobne struktury | `NamedTuple` |
| Ciężka pętla numeryczna / ML | `dataclass(slots=True)` lub raw NumPy arrays |
| Kontrakt API wyjściowy (response) | **Pydantic** — `model_dump_json()` |
| Konfiguracja aplikacji z env vars | `pydantic-settings` |
| Walidacja gołej listy/dicta | `TypeAdapter` |
| Schema dla Structured Outputs LLM-a | **Pydantic** — `model_json_schema()` |
| Polimorficzne payload-y (różne typy eventów) | **Pydantic** + Discriminated Union |

**Reguła kciuka: Pydantic na granicach systemu, dataclasses w środku.** To zasada "anti-entropy at the edges" — chaos jest zatrzymywany na ingressu i egressu, a w środku panuje porządek i prędkość.
